In [1]:
# ============================================================
# AUSTRALIA ENVIRONMENTAL MONITORING PROJECT — FULL SCRIPT
# Run this as ONE cell in Jupyter. Everything is self-contained.
# ============================================================

import ee
import geemap

# ---------- AUTHENTICATION ----------
try:
    ee.Initialize()
    print("✅ EE initialized")
except Exception:
    ee.Authenticate()
    ee.Initialize()
    print("✅ EE initialized after authentication")

# ---------- CONFIG ----------
DRIVE_FOLDER = 'GEE_Exports'

# ---------- DATASETS ----------
gaul = ee.FeatureCollection("FAO/GAUL_SIMPLIFIED_500m/2015/level0")
Aus = gaul.filter(ee.Filter.eq('ADM0_NAME', 'Australia'))
australia_geometry = Aus.geometry()

Modis = ee.ImageCollection("MODIS/061/MCD64A1")
NDVI = ee.ImageCollection("MODIS/061/MOD13A2")
hansen = ee.Image("UMD/hansen/global_forest_change_2023_v1_11")
suomi_viirs = ee.ImageCollection("NASA/LANCE/SNPP_VIIRS/C2")
lst = ee.ImageCollection("MODIS/061/MOD11A1")
lulc = ee.ImageCollection("MODIS/061/MCD12C1")

print("✅ Datasets loaded")


# ============================================================
# STEP 1 — BURNED AREA (MODIS MCD64A1, 2021)
# ============================================================
modis_filtered = Modis.filterDate('2021-01-01', '2021-12-31').filterBounds(Aus)
burned = modis_filtered.select('BurnDate')

def compute_burned_area(image):
    burn_date = image.select('BurnDate')
    burned_mask = burn_date.gt(0)
    area_image = ee.Image.pixelArea().updateMask(burned_mask)
    burned_area = area_image.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=australia_geometry,
        scale=500,
        maxPixels=1e13
    )
    burned_area_ha = ee.Number(burned_area.get('area')).divide(10000)
    return ee.Feature(None, {
        'burnDate': ee.Date(image.get('system:time_start')).format('YYYY-MM-dd'),
        'burnedAreaHectares': burned_area_ha
    })

burned_area_collection = ee.FeatureCollection(burned.map(compute_burned_area))

burn_vis_params = {
    'min': 30, 'max': 355,
    'palette': ['ffffcc', 'ffeda0', 'fed976', 'feb24c',
                'fd8d3c', 'fc4e2a', 'e31a1c', 'bd0026', '800026']
}
ba = burned.max().clip(Aus)          # <-- 'ba' defined HERE
print("✅ Step 1: Burned Area ready")

ee.batch.Export.table.toDrive(
    collection=burned_area_collection,
    description='Burned_Area_2021',
    folder=DRIVE_FOLDER,
    fileNamePrefix='Burned_Area_2021',
    fileFormat='CSV'
).start()


# ============================================================
# STEP 2 — ACTIVE FIRES (VIIRS)
# ============================================================
suomi_filtered = suomi_viirs.filterDate('2023-10-08', '2023-10-30').filterBounds(Aus)
clipped_dataset = suomi_filtered.map(lambda img: img.clip(Aus))

def categorize_fire(image):
    bright = image.select('Bright_ti4')
    category = ee.Image(0) \
        .where(bright.gte(260).And(bright.lt(360)), 1) \
        .where(bright.gte(361).And(bright.lt(520)), 2) \
        .where(bright.gte(521).And(bright.lt(680)), 3) \
        .where(bright.gte(681).And(bright.lt(840)), 4) \
        .where(bright.gte(841), 5)
    return category.rename('Category').copyProperties(image, ['system:time_start'])

merged_fire = clipped_dataset.map(categorize_fire).mosaic()

fire_points = merged_fire.reduceToVectors(
    geometry=australia_geometry,
    scale=500,
    geometryType='centroid',
    bestEffort=True,
    crs='EPSG:4326'
)

def add_color(feature):
    cat = feature.get('label')
    color = ee.String(ee.Algorithms.If(
        ee.Number(cat).eq(1), 'yellow',
        ee.Algorithms.If(ee.Number(cat).eq(2), 'orange',
        ee.Algorithms.If(ee.Number(cat).eq(3), 'red',
        ee.Algorithms.If(ee.Number(cat).eq(4), 'white', 'darkred')))))
    return feature.set({'Color': color})

colored_fire_points = fire_points.map(add_color)
print("✅ Step 2: Fire Points ready")

ee.batch.Export.table.toDrive(
    collection=colored_fire_points,
    description='Categorized_FirePixels_Australia',
    folder=DRIVE_FOLDER,
    fileFormat='CSV',
    selectors=['longitude', 'latitude', 'label', 'Color']
).start()


# ============================================================
# STEP 3 — LAND SURFACE TEMPERATURE (2019)
# ============================================================
lst_filtered = lst.filterDate('2019-01-01', '2019-12-31').filterBounds(Aus)
lst_day = lst_filtered.select('LST_Day_1km')

def kelvin_to_celsius(img):
    return img.multiply(0.02).subtract(273.15).copyProperties(img, ['system:time_start'])

mean_lst = lst_day.map(kelvin_to_celsius).mean().clip(Aus)
lst_vis_params = {'min': 10, 'max': 45,
                  'palette': ['blue', 'limegreen', 'yellow', 'darkorange', 'red']}
print("✅ Step 3: LST ready")

ee.batch.Export.image.toDrive(
    image=mean_lst,
    description='LandSurfTemp_Aus_2019',
    folder=DRIVE_FOLDER,
    scale=1000,
    crs='EPSG:4326',
    fileFormat='GeoTIFF',
    maxPixels=1e10,
    region=australia_geometry
).start()


# ============================================================
# STEP 4 — LAND USE / LAND COVER (2022)
# ============================================================
lulc_img = lulc.filterDate('2022-01-01', '2022-12-31').filterBounds(Aus).first().clip(Aus)

lookup_in  = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
lookup_out = [0, 1, 1, 1, 1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 6, 7]

classified_land_cover = lulc_img.select('Majority_Land_Cover_Type_1') \
                                .remap(lookup_in, lookup_out, 7) \
                                .rename('classified')

lulc_vis_params = {'min': 0, 'max': 7,
                   'palette': ['blue', 'green', 'yellow', 'orange',
                               'red', 'gray', 'white', 'cyan']}
print("✅ Step 4: Land Cover ready")

ee.batch.Export.image.toDrive(
    image=classified_land_cover,
    description='LULC_Aus_2022',
    folder=DRIVE_FOLDER,
    maxPixels=1e13,
    scale=500,
    region=australia_geometry,
    fileFormat='GeoTIFF',
    crs='EPSG:4326'
).start()


# ============================================================
# STEP 5 — NDVI (Annual mean, 2010–2021)
# ============================================================
ndvi_filtered = NDVI.select('NDVI').filterBounds(Aus)
years = ee.List.sequence(2010, 2021, 1)
months = ee.List.sequence(1, 12, 1)

def make_monthly(y):
    def for_month(m):
        img = ndvi_filtered \
            .filter(ee.Filter.calendarRange(y, y, 'year')) \
            .filter(ee.Filter.calendarRange(m, m, 'month')) \
            .mean() \
            .set('system:time_start', ee.Date.fromYMD(y, m, 1).millis())
        return img
    return months.map(for_month)

ndvi_monthly = ee.ImageCollection.fromImages(years.map(make_monthly).flatten())
ndvi_annual_mean = ndvi_monthly.mean().clip(Aus)

ndvi_vis_params = {'min': -2000, 'max': 10000, 'palette': ['blue', 'white', 'green']}
print("✅ Step 5: NDVI ready")

ee.batch.Export.image.toDrive(
    image=ndvi_annual_mean,
    description='NDVI_Annual_Mean_Aus',
    folder=DRIVE_FOLDER,
    scale=1000,
    crs='EPSG:4326',
    fileFormat='GeoTIFF',
    maxPixels=1e10,
    region=australia_geometry
).start()


# ============================================================
# STEP 6 — FOREST LOSS (Hansen 2020–2022)
# ============================================================
loss_year = hansen.select(['lossyear'])
forest_loss_period = loss_year.gte(2020 - 2000).And(loss_year.lte(2022 - 2000))
forest_loss_masked = hansen.select(['loss']).updateMask(forest_loss_period).clip(Aus)
forest_loss_vis = {'min': 0, 'max': 1, 'palette': ['red']}
print("✅ Step 6: Forest Loss ready")

ee.batch.Export.image.toDrive(
    image=forest_loss_masked,
    description='Forest_Loss_Aus_2020_2022',
    folder=DRIVE_FOLDER,
    scale=500,
    crs='EPSG:4326',
    fileFormat='GeoTIFF',
    maxPixels=1e10,
    region=australia_geometry
).start()


# ============================================================
# PNG EXPORTS (high quality, via geemap)
# ============================================================
def export_png(image, vis, filename, scale=1000):
    try:
        geemap.ee_export_image(
            image.visualize(**vis),
            filename=filename,
            scale=scale,
            region=australia_geometry,
            file_per_band=False,
            crs='EPSG:4326'
        )
        print(f"✅ PNG: {filename}")
    except Exception as e:
        print(f"⚠️ PNG failed ({filename}): {e}")

print("\n🎨 Exporting PNG maps...")
export_png(ba, burn_vis_params, 'Burned_Area_2021.png', 2000)
export_png(merged_fire, {'min':1,'max':5,
            'palette':['yellow','orange','red','white','darkred']},
           'Fire_Categories_2023.png', 2000)
export_png(mean_lst, lst_vis_params, 'LST_Australia_2019.png', 2000)
export_png(classified_land_cover, lulc_vis_params, 'LULC_Australia_2022.png', 2000)
export_png(ndvi_annual_mean, ndvi_vis_params, 'NDVI_Australia_Mean.png', 2000)
export_png(forest_loss_masked, forest_loss_vis, 'Forest_Loss_2020_2022.png', 2000)


# ============================================================
# TILE URLs FOR WEBGIS  (fixed — must call .getInfo())
# ============================================================
def get_tile_url(image, vis, name):
    vis_img = image.visualize(**vis)
    map_id = vis_img.getMapId()      # <-- dict on client side
    url = map_id['tile_fetcher'].url_format
    print(f"\n🗺️ {name}:\n   {url}")
    return url

print("\n" + "="*60)
print("WEBGIS TILE URLs (paste into Leaflet / OpenLayers)")
print("="*60)

tile_urls = {
    'burned':      get_tile_url(ba,                    burn_vis_params, 'Burned Area 2021'),
    'fire':        get_tile_url(merged_fire,           {'min':1,'max':5,
                        'palette':['yellow','orange','red','white','darkred']},
                        'Fire Categories 2023'),
    'lst':         get_tile_url(mean_lst,              lst_vis_params,  'LST 2019'),
    'lulc':        get_tile_url(classified_land_cover, lulc_vis_params, 'Land Cover 2022'),
    'ndvi':        get_tile_url(ndvi_annual_mean,      ndvi_vis_params, 'NDVI Mean'),
    'forest_loss': get_tile_url(forest_loss_masked,    forest_loss_vis,'Forest Loss 2020–2022'),
}


# ============================================================
# INTERACTIVE MAP (Jupyter display)
# ============================================================
m = geemap.Map(center=[-25, 134], zoom=4)
m.add_basemap('HYBRID')
m.addLayer(ba, burn_vis_params, '🔥 Burned Area 2021')
m.addLayer(mean_lst, lst_vis_params, '🌡️ LST 2019')
m.addLayer(classified_land_cover, lulc_vis_params, '🌍 Land Cover 2022')
m.addLayer(ndvi_annual_mean, ndvi_vis_params, '🌿 NDVI Mean')
m.addLayer(forest_loss_masked, forest_loss_vis, '🌲 Forest Loss 2020–2022')
m.addLayer(merged_fire, {'min':1,'max':5,
    'palette':['yellow','orange','red','white','darkred']}, '🔥 Fire Categories 2023')
m.addLayer(Aus, {'color': 'black'}, 'Australia Boundary')
m.addLayerControl()

print("\n✅ Everything complete. Displaying map...")
m




*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python
/opt/anaconda3/envs/geop_env/lib/python3.13/site-packages/ee/deprecation.py:207: DeprecationWarning: 

Attention required for UMD/hansen/global_forest_change_2023_v1_11! You are using a deprecated asset.
To make sure your code keeps working, please update it.
Learn more: https://developers.google.com/earth-engine/datasets/catalog/UMD_hansen_global_forest_change_2023_v1_11

  warnings.warn(warning, category=DeprecationWarning)


✅ EE initialized
✅ Datasets loaded
✅ Step 1: Burned Area ready
✅ Step 2: Fire Points ready
✅ Step 3: LST ready
✅ Step 4: Land Cover ready
✅ Step 5: NDVI ready
✅ Step 6: Forest Loss ready

🎨 Exporting PNG maps...
The filename must end with .tif
✅ PNG: Burned_Area_2021.png
The filename must end with .tif
✅ PNG: Fire_Categories_2023.png
The filename must end with .tif
✅ PNG: LST_Australia_2019.png
The filename must end with .tif
✅ PNG: LULC_Australia_2022.png
The filename must end with .tif
✅ PNG: NDVI_Australia_Mean.png
The filename must end with .tif
✅ PNG: Forest_Loss_2020_2022.png

WEBGIS TILE URLs (paste into Leaflet / OpenLayers)

🗺️ Burned Area 2021:
   https://earthengine.googleapis.com/v1/projects/99406868390/maps/8a80d6e7b9feb4c25a4ebac0d15a5101-ec12efde6026822ca8ff01695efa5fdb/tiles/{z}/{x}/{y}

🗺️ Fire Categories 2023:
   https://earthengine.googleapis.com/v1/projects/99406868390/maps/c3d11bee8ca70c1f97a30e779307d9dd-bbf5f7c890c7319c22f19b32d8498d27/tiles/{z}/{x}/{y}

🗺️ LST 2

Map(center=[-25, 134], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(ch…

In [4]:
# ============================================================
# AUSTRALIA ENVIRONMENTAL MONITORING PROJECT — FULL UPDATED SCRIPT (v3)
# Run as ONE cell in Jupyter. Self-contained.
# ============================================================

import ee
import geemap
import requests
import os
import json
from requests.exceptions import HTTPError

# ============================================================
# 0. AUTHENTICATION
# ============================================================
try:
    ee.Initialize()
    print("✅ EE initialized")
except Exception:
    ee.Authenticate()
    ee.Initialize()
    print("✅ EE initialized after authentication")

# ============================================================
# 1. CONFIG
# ============================================================
DRIVE_FOLDER = 'GEE_Exports'
PNG_DIR = 'exports_png'
os.makedirs(PNG_DIR, exist_ok=True)

# ============================================================
# 2. DATASETS
# ============================================================
gaul = ee.FeatureCollection("FAO/GAUL_SIMPLIFIED_500m/2015/level0")
Aus = gaul.filter(ee.Filter.eq('ADM0_NAME', 'Australia'))
australia_geometry = Aus.geometry()

Modis       = ee.ImageCollection("MODIS/061/MCD64A1")
NDVI        = ee.ImageCollection("MODIS/061/MOD13A2")
hansen      = ee.Image("UMD/hansen/global_forest_change_2023_v1_11")
suomi_viirs = ee.ImageCollection("NASA/LANCE/SNPP_VIIRS/C2")
lst         = ee.ImageCollection("MODIS/061/MOD11A1")
lulc        = ee.ImageCollection("MODIS/061/MCD12C1")

print("✅ Datasets loaded")


# ============================================================
# STEP 1 — BURNED AREA (MODIS MCD64A1, 2021)
# ============================================================
modis_filtered = Modis.filterDate('2021-01-01', '2021-12-31').filterBounds(Aus)
burned = modis_filtered.select('BurnDate')

def compute_burned_area(image):
    burn_date = image.select('BurnDate')
    burned_mask = burn_date.gt(0)
    area_image = ee.Image.pixelArea().updateMask(burned_mask)
    burned_area = area_image.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=australia_geometry,
        scale=500,
        maxPixels=1e13
    )
    burned_area_ha = ee.Number(burned_area.get('area')).divide(10000)
    return ee.Feature(None, {
        'burnDate': ee.Date(image.get('system:time_start')).format('YYYY-MM-dd'),
        'burnedAreaHectares': burned_area_ha
    })

burned_area_collection = ee.FeatureCollection(burned.map(compute_burned_area))

burn_vis_params = {
    'min': 30, 'max': 355,
    'palette': ['ffffcc', 'ffeda0', 'fed976', 'feb24c',
                'fd8d3c', 'fc4e2a', 'e31a1c', 'bd0026', '800026']
}
ba = burned.max().clip(Aus)
print("✅ Step 1: Burned Area ready")

ee.batch.Export.table.toDrive(
    collection=burned_area_collection,
    description='Burned_Area_2021',
    folder=DRIVE_FOLDER,
    fileNamePrefix='Burned_Area_2021',
    fileFormat='CSV'
).start()


# ============================================================
# STEP 2 — ACTIVE FIRES (VIIRS) — CORRECTED
# ============================================================
viirs = suomi_viirs.filterDate('2023-10-08', '2023-10-30').filterBounds(Aus)
print(f"✅ Step 2: VIIRS scenes in AOI/period: {viirs.size().getInfo()}")

def mask_and_classify(img):
    """Bin Bright_ti4 (Kelvin) into 5 realistic fire-temperature classes."""
    bright = img.select('Bright_ti4')
    category = (
        ee.Image(0)
        .where(bright.gte(300).And(bright.lt(320)), 1)   # cool
        .where(bright.gte(320).And(bright.lt(340)), 2)   # moderate
        .where(bright.gte(340).And(bright.lt(360)), 3)   # hot
        .where(bright.gte(360).And(bright.lt(380)), 4)   # very hot
        .where(bright.gte(380), 5)                       # extreme
    )
    category = category.updateMask(category.gt(0)).rename('Category')
    return category.copyProperties(img, ['system:time_start'])

classified_viirs = viirs.map(mask_and_classify)

merged_fire = classified_viirs.max().clip(Aus)
fire_count  = classified_viirs.count().clip(Aus).rename('fire_count')

# Reduce to vector points
fire_points = merged_fire.reduceToVectors(
    geometry=australia_geometry,
    scale=1000,
    geometryType='centroid',
    labelProperty='Category',
    bestEffort=True,
    crs='EPSG:4326'
)

def classify_point(feature):
    cat = ee.Number(feature.get('Category'))
    label = ee.String(ee.Algorithms.If(
        cat.eq(1), 'Cool (300-320K)',
        ee.Algorithms.If(
            cat.eq(2), 'Moderate (320-340K)',
            ee.Algorithms.If(
                cat.eq(3), 'Hot (340-360K)',
                ee.Algorithms.If(
                    cat.eq(4), 'Very Hot (360-380K)',
                    'Extreme (>380K)'
                )
            )
        )
    ))
    color = ee.String(ee.Algorithms.If(
        cat.eq(1), 'ffff00',
        ee.Algorithms.If(
            cat.eq(2), 'ffa500',
            ee.Algorithms.If(
                cat.eq(3), 'ff0000',
                ee.Algorithms.If(
                    cat.eq(4), 'ffffff',
                    '8b0000'
                )
            )
        )
    ))
    return feature.set({'Category': cat, 'Temp_Class': label, 'Color': color})

colored_fire_points = fire_points.map(classify_point)
print(f"✅ Step 2: Fire points ready (~{colored_fire_points.size().getInfo()} pts)")

fire_vis_params = {
    'min': 1, 'max': 5,
    'palette': ['ffff00', 'ffa500', 'ff0000', 'ffffff', '8b0000']
}
fire_count_vis = {
    'min': 0, 'max': 20,
    'palette': ['000000', '440154', '3b528b', '21918c', '5ec962', 'fde725']
}

ee.batch.Export.table.toDrive(
    collection=colored_fire_points,
    description='Categorized_FirePixels_Australia',
    folder=DRIVE_FOLDER,
    fileFormat='CSV',
    selectors=['longitude', 'latitude', 'Category', 'Temp_Class', 'Color']
).start()

ee.batch.Export.image.toDrive(
    image=fire_count,
    description='Fire_Count_Aus_2023',
    folder=DRIVE_FOLDER,
    scale=1000,
    crs='EPSG:4326',
    region=australia_geometry,
    fileFormat='GeoTIFF',
    maxPixels=1e10
).start()
print("📤 Fire CSVs + GeoTIFF queued")


# ============================================================
# STEP 3 — LAND SURFACE TEMPERATURE (2019)
# ============================================================
lst_filtered = lst.filterDate('2019-01-01', '2019-12-31').filterBounds(Aus)
lst_day = lst_filtered.select('LST_Day_1km')

def kelvin_to_celsius(img):
    return img.multiply(0.02).subtract(273.15).copyProperties(img, ['system:time_start'])

mean_lst = lst_day.map(kelvin_to_celsius).mean().clip(Aus)
lst_vis_params = {'min': 10, 'max': 45,
                  'palette': ['blue', 'limegreen', 'yellow', 'darkorange', 'red']}
print("✅ Step 3: LST ready")

ee.batch.Export.image.toDrive(
    image=mean_lst,
    description='LandSurfTemp_Aus_2019',
    folder=DRIVE_FOLDER,
    scale=1000,
    crs='EPSG:4326',
    fileFormat='GeoTIFF',
    maxPixels=1e10,
    region=australia_geometry
).start()


# ============================================================
# STEP 4 — LAND USE / LAND COVER (2022)
# ============================================================
lulc_img = lulc.filterDate('2022-01-01', '2022-12-31').filterBounds(Aus).first().clip(Aus)

lookup_in  = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
lookup_out = [0, 1, 1, 1, 1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 6, 7]

classified_land_cover = (
    lulc_img.select('Majority_Land_Cover_Type_1')
            .remap(lookup_in, lookup_out, 7)
            .rename('classified')
)

lulc_vis_params = {'min': 0, 'max': 7,
                   'palette': ['blue', 'green', 'yellow', 'orange',
                               'red', 'gray', 'white', 'cyan']}
print("✅ Step 4: Land Cover ready")

ee.batch.Export.image.toDrive(
    image=classified_land_cover,
    description='LULC_Aus_2022',
    folder=DRIVE_FOLDER,
    maxPixels=1e13,
    scale=500,
    region=australia_geometry,
    fileFormat='GeoTIFF',
    crs='EPSG:4326'
).start()

# LULC area table
pixel_area = ee.Image.pixelArea()
def compute_area(lc_value):
    mask = classified_land_cover.eq(lc_value)
    area = mask.multiply(pixel_area).reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=australia_geometry,
        scale=500,
        maxPixels=1e13
    )
    return ee.Number(area.get('classified')).divide(1e6)

lc_names = ['Water', 'Forest', 'Shrubs', 'Grass',
            'Cropland', 'Mixed Land', 'Sparse', 'Snow/Ice']
area_features = [
    ee.Feature(None, {'Land Cover Type': lc_names[i], 'Area (sq km)': compute_area(i)})
    for i in range(8)
]
ee.batch.Export.table.toDrive(
    collection=ee.FeatureCollection(area_features),
    description='LandCoverArea_Australia_2022',
    folder=DRIVE_FOLDER,
    fileFormat='CSV'
).start()


# ============================================================
# STEP 5 — NDVI (Annual mean, 2010–2021) — INT16 SAFE
# ============================================================
ndvi_filtered = NDVI.select('NDVI').filterBounds(Aus)
years  = ee.List.sequence(2010, 2021, 1)
months = ee.List.sequence(1, 12, 1)

def make_monthly(y):
    def for_month(m):
        return ndvi_filtered \
            .filter(ee.Filter.calendarRange(y, y, 'year')) \
            .filter(ee.Filter.calendarRange(m, m, 'month')) \
            .mean() \
            .set('system:time_start', ee.Date.fromYMD(y, m, 1).millis())
    return months.map(for_month)

ndvi_monthly = ee.ImageCollection.fromImages(years.map(make_monthly).flatten())

# Cast to Int16 (avoids thumbnail-service Float32 issues) and clip
ndvi_annual_mean     = ndvi_monthly.mean().clip(Aus)                     # float, for stats
ndvi_annual_mean_int = ndvi_annual_mean.toInt16()                        # int, for PNG/tiles

# Palette tightened to typical Australian NDVI range (more contrast, smaller PNG)
ndvi_vis_params = {
    'min': 0,
    'max': 8000,
    'palette': ['#8B4513', '#FFFFE0', '#ADFF2F', '#228B22', '#006400']
}
print("✅ Step 5: NDVI ready")

ee.batch.Export.image.toDrive(
    image=ndvi_annual_mean,
    description='NDVI_Annual_Mean_Aus',
    folder=DRIVE_FOLDER,
    scale=1000,
    crs='EPSG:4326',
    fileFormat='GeoTIFF',
    maxPixels=1e10,
    region=australia_geometry
).start()


# ============================================================
# STEP 6 — FOREST LOSS (Hansen 2020–2022)
# ============================================================
loss_year = hansen.select(['lossyear'])
forest_loss_period = loss_year.gte(2020 - 2000).And(loss_year.lte(2022 - 2000))
forest_loss_masked = hansen.select(['loss']).updateMask(forest_loss_period).clip(Aus)
forest_loss_vis = {'min': 0, 'max': 1, 'palette': ['red']}
print("✅ Step 6: Forest Loss ready")

ee.batch.Export.image.toDrive(
    image=forest_loss_masked,
    description='Forest_Loss_Aus_2020_2022',
    folder=DRIVE_FOLDER,
    scale=500,
    crs='EPSG:4326',
    fileFormat='GeoTIFF',
    maxPixels=1e10,
    region=australia_geometry
).start()


# ============================================================
# ROBUST PNG EXPORT  (bbox + auto-retry)
# ============================================================
def _bbox_from_geometry(geom, pad_deg=0.5):
    """Return [minLon, minLat, maxLon, maxLat] for a geometry."""
    coords = geom.bounds().getInfo()['coordinates'][0]
    xs = [c[0] for c in coords]
    ys = [c[1] for c in coords]
    return [min(xs) - pad_deg, min(ys) - pad_deg,
            max(xs) + pad_deg, max(ys) + pad_deg]


def export_png(image, vis, filename, region=australia_geometry,
               scale=2000, dimensions=None, max_retries=3):
    """
    Robust PNG export using getThumbURL.

    - Uses bbox instead of polygon (avoids thumbnail-service 400s).
    - Retries with progressively smaller payloads if the server rejects.
    """
    vis_img = image.visualize(**vis)
    bbox = _bbox_from_geometry(region)

    attempts = []
    if dimensions:
        attempts.append({'region': bbox, 'format': 'png', 'dimensions': dimensions})
    else:
        attempts.append({'region': bbox, 'format': 'png', 'scale': scale})
        attempts.append({'region': bbox, 'format': 'png', 'scale': scale * 2})
        attempts.append({'region': bbox, 'format': 'png', 'dimensions': 1024})

    last_err = None
    for i, params in enumerate(attempts[:max_retries], 1):
        try:
            url = vis_img.getThumbURL(params)
            r = requests.get(url, stream=True, timeout=300)
            r.raise_for_status()
            with open(filename, 'wb') as f:
                for chunk in r.iter_content(8192):
                    f.write(chunk)
            print(f"✅ PNG: {filename}  (attempt {i}, params={params})")
            return True
        except HTTPError as e:
            last_err = e
            print(f"⚠️ Attempt {i} failed for {filename}: {e}")
        except Exception as e:
            last_err = e
            print(f"⚠️ Attempt {i} failed for {filename}: {e}")

    print(f"❌ Could not export {filename}: {last_err}")
    return False


print("\n🎨 Exporting PNG maps...")
export_png(ba,                    burn_vis_params, f'{PNG_DIR}/Burned_Area_2021.png',      scale=2000)
export_png(merged_fire,           fire_vis_params, f'{PNG_DIR}/Fire_Categories_2023.png',  scale=2000)
export_png(fire_count,            fire_count_vis,  f'{PNG_DIR}/Fire_Count_2023.png',       scale=2000)
export_png(mean_lst,              lst_vis_params,  f'{PNG_DIR}/LST_Australia_2019.png',    scale=2000)
export_png(classified_land_cover, lulc_vis_params, f'{PNG_DIR}/LULC_Australia_2022.png',   scale=2000)
export_png(ndvi_annual_mean_int,  ndvi_vis_params, f'{PNG_DIR}/NDVI_Australia_Mean.png',   scale=2000)
export_png(forest_loss_masked,    forest_loss_vis, f'{PNG_DIR}/Forest_Loss_2020_2022.png', scale=2000)


# ============================================================
# WEBGIS TILE URLs
# ============================================================
def get_tile_url(image, vis, name):
    map_id = image.visualize(**vis).getMapId()
    url = map_id['tile_fetcher'].url_format
    print(f"\n🗺️ {name}:\n   {url}")
    return url

print("\n" + "=" * 60)
print("WEBGIS TILE URLs (paste into Leaflet / OpenLayers)")
print("=" * 60)

tile_urls = {
    'burned':      get_tile_url(ba,                    burn_vis_params,   'Burned Area 2021'),
    'fire':        get_tile_url(merged_fire,           fire_vis_params,   'Fire Categories 2023'),
    'fire_count':  get_tile_url(fire_count,            fire_count_vis,    'Fire Detection Count 2023'),
    'lst':         get_tile_url(mean_lst,              lst_vis_params,    'LST 2019'),
    'lulc':        get_tile_url(classified_land_cover, lulc_vis_params,   'Land Cover 2022'),
    'ndvi':        get_tile_url(ndvi_annual_mean_int,  ndvi_vis_params,   'NDVI Mean'),
    'forest_loss': get_tile_url(forest_loss_masked,    forest_loss_vis,   'Forest Loss 2020–2022'),
}

with open('tile_urls.json', 'w') as f:
    json.dump(tile_urls, f, indent=2)
print("\n💾 Saved tile_urls.json")


# ============================================================
# INTERACTIVE MAP (Jupyter display)
# ============================================================
m = geemap.Map(center=[-25, 134], zoom=4)
m.add_basemap('HYBRID')

m.addLayer(ba,                    burn_vis_params,   '🔥 Burned Area 2021')
m.addLayer(mean_lst,              lst_vis_params,    '🌡️ LST 2019')
m.addLayer(classified_land_cover, lulc_vis_params,   '🌍 Land Cover 2022')
m.addLayer(ndvi_annual_mean_int,  ndvi_vis_params,   '🌿 NDVI Mean')
m.addLayer(forest_loss_masked,    forest_loss_vis,   '🌲 Forest Loss 2020–2022')
m.addLayer(merged_fire,           fire_vis_params,   '🔥 Fire Categories 2023')
m.addLayer(fire_count,            fire_count_vis,    '🔥 Fire Detection Count 2023')
m.addLayer(Aus,                   {'color': 'black'}, 'Australia Boundary')

m.addLayerControl()

legend_dict = {
    'Cool (300–320K)':     'ffff00',
    'Moderate (320–340K)': 'ffa500',
    'Hot (340–360K)':      'ff0000',
    'Very Hot (360–380K)': 'ffffff',
    'Extreme (>380K)':     '8b0000',
}
m.add_legend(title='Fire Temperature Class', legend_dict=legend_dict)

print("\n✅ ALL STEPS COMPLETE")
print(f"   📁 PNGs: ./{PNG_DIR}/")
print(f"   📁 Drive: {DRIVE_FOLDER}")
print(f"   📁 Tile URLs: tile_urls.json")
m




✅ EE initialized
✅ Datasets loaded
✅ Step 1: Burned Area ready
✅ Step 2: VIIRS scenes in AOI/period: 22
✅ Step 2: Fire points ready (~38572 pts)
📤 Fire CSVs + GeoTIFF queued
✅ Step 3: LST ready
✅ Step 4: Land Cover ready
✅ Step 5: NDVI ready
✅ Step 6: Forest Loss ready

🎨 Exporting PNG maps...
✅ PNG: exports_png/Burned_Area_2021.png  (attempt 1, params={'region': [112.42108988316402, -44.24058444952691, 159.60954806417263, -8.642471596906729], 'format': 'png', 'scale': 2000})
✅ PNG: exports_png/Fire_Categories_2023.png  (attempt 1, params={'region': [112.42108988316402, -44.24058444952691, 159.60954806417263, -8.642471596906729], 'format': 'png', 'scale': 2000})
✅ PNG: exports_png/Fire_Count_2023.png  (attempt 1, params={'region': [112.42108988316402, -44.24058444952691, 159.60954806417263, -8.642471596906729], 'format': 'png', 'scale': 2000})
⚠️ Attempt 1 failed for exports_png/LST_Australia_2019.png: HTTPSConnectionPool(host='earthengine.googleapis.com', port=443): Read timed out. (r

Map(center=[-25, 134], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(ch…

In [5]:
# ============================================================
# AUSTRALIA ENVIRONMENTAL MONITORING PROJECT — FULL SCRIPT (v4)
# Run as ONE cell in Jupyter. Self-contained.
# Includes: black-background PNGs, auto-retry, title/border styling.
# ============================================================

import ee
import geemap
import requests
import os
import json
import glob
from requests.exceptions import HTTPError
from PIL import Image, ImageDraw, ImageFont

# ============================================================
# 0. AUTHENTICATION
# ============================================================
try:
    ee.Initialize()
    print("✅ EE initialized")
except Exception:
    ee.Authenticate()
    ee.Initialize()
    print("✅ EE initialized after authentication")

# ============================================================
# 1. CONFIG
# ============================================================
DRIVE_FOLDER = 'GEE_Exports'
PNG_DIR      = 'exports_png'          # raw PNGs from GEE (black bg)
PNG_FINAL_DIR = 'exports_final'       # stylized (title + border)
os.makedirs(PNG_DIR, exist_ok=True)
os.makedirs(PNG_FINAL_DIR, exist_ok=True)

# ============================================================
# 2. DATASETS
# ============================================================
gaul = ee.FeatureCollection("FAO/GAUL_SIMPLIFIED_500m/2015/level0")
Aus = gaul.filter(ee.Filter.eq('ADM0_NAME', 'Australia'))
australia_geometry = Aus.geometry()

Modis       = ee.ImageCollection("MODIS/061/MCD64A1")
NDVI        = ee.ImageCollection("MODIS/061/MOD13A2")
hansen      = ee.Image("UMD/hansen/global_forest_change_2023_v1_11")
suomi_viirs = ee.ImageCollection("NASA/LANCE/SNPP_VIIRS/C2")
lst         = ee.ImageCollection("MODIS/061/MOD11A1")
lulc        = ee.ImageCollection("MODIS/061/MCD12C1")

print("✅ Datasets loaded")


# ============================================================
# STEP 1 — BURNED AREA (MODIS MCD64A1, 2021)
# ============================================================
modis_filtered = Modis.filterDate('2021-01-01', '2021-12-31').filterBounds(Aus)
burned = modis_filtered.select('BurnDate')

def compute_burned_area(image):
    burn_date = image.select('BurnDate')
    burned_mask = burn_date.gt(0)
    area_image = ee.Image.pixelArea().updateMask(burned_mask)
    burned_area = area_image.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=australia_geometry,
        scale=500,
        maxPixels=1e13
    )
    burned_area_ha = ee.Number(burned_area.get('area')).divide(10000)
    return ee.Feature(None, {
        'burnDate': ee.Date(image.get('system:time_start')).format('YYYY-MM-dd'),
        'burnedAreaHectares': burned_area_ha
    })

burned_area_collection = ee.FeatureCollection(burned.map(compute_burned_area))

burn_vis_params = {
    'min': 30, 'max': 355,
    'palette': ['ffffcc', 'ffeda0', 'fed976', 'feb24c',
                'fd8d3c', 'fc4e2a', 'e31a1c', 'bd0026', '800026']
}
ba = burned.max().clip(Aus)
print("✅ Step 1: Burned Area ready")

ee.batch.Export.table.toDrive(
    collection=burned_area_collection,
    description='Burned_Area_2021',
    folder=DRIVE_FOLDER,
    fileNamePrefix='Burned_Area_2021',
    fileFormat='CSV'
).start()


# ============================================================
# STEP 2 — ACTIVE FIRES (VIIRS) — CORRECTED
# ============================================================
viirs = suomi_viirs.filterDate('2023-10-08', '2023-10-30').filterBounds(Aus)
print(f"✅ Step 2: VIIRS scenes in AOI/period: {viirs.size().getInfo()}")

def mask_and_classify(img):
    bright = img.select('Bright_ti4')
    category = (
        ee.Image(0)
        .where(bright.gte(300).And(bright.lt(320)), 1)
        .where(bright.gte(320).And(bright.lt(340)), 2)
        .where(bright.gte(340).And(bright.lt(360)), 3)
        .where(bright.gte(360).And(bright.lt(380)), 4)
        .where(bright.gte(380), 5)
    )
    category = category.updateMask(category.gt(0)).rename('Category')
    return category.copyProperties(img, ['system:time_start'])

classified_viirs = viirs.map(mask_and_classify)

merged_fire = classified_viirs.max().clip(Aus)
fire_count  = classified_viirs.count().clip(Aus).rename('fire_count')

fire_points = merged_fire.reduceToVectors(
    geometry=australia_geometry,
    scale=1000,
    geometryType='centroid',
    labelProperty='Category',
    bestEffort=True,
    crs='EPSG:4326'
)

def classify_point(feature):
    cat = ee.Number(feature.get('Category'))
    label = ee.String(ee.Algorithms.If(
        cat.eq(1), 'Cool (300-320K)',
        ee.Algorithms.If(
            cat.eq(2), 'Moderate (320-340K)',
            ee.Algorithms.If(
                cat.eq(3), 'Hot (340-360K)',
                ee.Algorithms.If(
                    cat.eq(4), 'Very Hot (360-380K)',
                    'Extreme (>380K)'
                )
            )
        )
    ))
    color = ee.String(ee.Algorithms.If(
        cat.eq(1), 'ffff00',
        ee.Algorithms.If(
            cat.eq(2), 'ffa500',
            ee.Algorithms.If(
                cat.eq(3), 'ff0000',
                ee.Algorithms.If(
                    cat.eq(4), 'ffffff',
                    '8b0000'
                )
            )
        )
    ))
    return feature.set({'Category': cat, 'Temp_Class': label, 'Color': color})

colored_fire_points = fire_points.map(classify_point)
print(f"✅ Step 2: Fire points ready (~{colored_fire_points.size().getInfo()} pts)")

fire_vis_params = {
    'min': 1, 'max': 5,
    'palette': ['ffff00', 'ffa500', 'ff0000', 'ffffff', '8b0000']
}
fire_count_vis = {
    'min': 0, 'max': 20,
    'palette': ['000000', '440154', '3b528b', '21918c', '5ec962', 'fde725']
}

ee.batch.Export.table.toDrive(
    collection=colored_fire_points,
    description='Categorized_FirePixels_Australia',
    folder=DRIVE_FOLDER,
    fileFormat='CSV',
    selectors=['longitude', 'latitude', 'Category', 'Temp_Class', 'Color']
).start()

ee.batch.Export.image.toDrive(
    image=fire_count,
    description='Fire_Count_Aus_2023',
    folder=DRIVE_FOLDER,
    scale=1000,
    crs='EPSG:4326',
    region=australia_geometry,
    fileFormat='GeoTIFF',
    maxPixels=1e10
).start()
print("📤 Fire CSVs + GeoTIFF queued")


# ============================================================
# STEP 3 — LAND SURFACE TEMPERATURE (2019)
# ============================================================
lst_filtered = lst.filterDate('2019-01-01', '2019-12-31').filterBounds(Aus)
lst_day = lst_filtered.select('LST_Day_1km')

def kelvin_to_celsius(img):
    return img.multiply(0.02).subtract(273.15).copyProperties(img, ['system:time_start'])

mean_lst = lst_day.map(kelvin_to_celsius).mean().clip(Aus)
lst_vis_params = {'min': 10, 'max': 45,
                  'palette': ['blue', 'limegreen', 'yellow', 'darkorange', 'red']}
print("✅ Step 3: LST ready")

ee.batch.Export.image.toDrive(
    image=mean_lst,
    description='LandSurfTemp_Aus_2019',
    folder=DRIVE_FOLDER,
    scale=1000,
    crs='EPSG:4326',
    fileFormat='GeoTIFF',
    maxPixels=1e10,
    region=australia_geometry
).start()


# ============================================================
# STEP 4 — LAND USE / LAND COVER (2022)
# ============================================================
lulc_img = lulc.filterDate('2022-01-01', '2022-12-31').filterBounds(Aus).first().clip(Aus)

lookup_in  = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
lookup_out = [0, 1, 1, 1, 1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 6, 7]

classified_land_cover = (
    lulc_img.select('Majority_Land_Cover_Type_1')
            .remap(lookup_in, lookup_out, 7)
            .rename('classified')
)

lulc_vis_params = {'min': 0, 'max': 7,
                   'palette': ['blue', 'green', 'yellow', 'orange',
                               'red', 'gray', 'white', 'cyan']}
print("✅ Step 4: Land Cover ready")

ee.batch.Export.image.toDrive(
    image=classified_land_cover,
    description='LULC_Aus_2022',
    folder=DRIVE_FOLDER,
    maxPixels=1e13,
    scale=500,
    region=australia_geometry,
    fileFormat='GeoTIFF',
    crs='EPSG:4326'
).start()

pixel_area = ee.Image.pixelArea()
def compute_area(lc_value):
    mask = classified_land_cover.eq(lc_value)
    area = mask.multiply(pixel_area).reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=australia_geometry,
        scale=500,
        maxPixels=1e13
    )
    return ee.Number(area.get('classified')).divide(1e6)

lc_names = ['Water', 'Forest', 'Shrubs', 'Grass',
            'Cropland', 'Mixed Land', 'Sparse', 'Snow/Ice']
area_features = [
    ee.Feature(None, {'Land Cover Type': lc_names[i], 'Area (sq km)': compute_area(i)})
    for i in range(8)
]
ee.batch.Export.table.toDrive(
    collection=ee.FeatureCollection(area_features),
    description='LandCoverArea_Australia_2022',
    folder=DRIVE_FOLDER,
    fileFormat='CSV'
).start()


# ============================================================
# STEP 5 — NDVI (Annual mean, 2010–2021) — INT16 SAFE
# ============================================================
ndvi_filtered = NDVI.select('NDVI').filterBounds(Aus)
years  = ee.List.sequence(2010, 2021, 1)
months = ee.List.sequence(1, 12, 1)

def make_monthly(y):
    def for_month(m):
        return ndvi_filtered \
            .filter(ee.Filter.calendarRange(y, y, 'year')) \
            .filter(ee.Filter.calendarRange(m, m, 'month')) \
            .mean() \
            .set('system:time_start', ee.Date.fromYMD(y, m, 1).millis())
    return months.map(for_month)

ndvi_monthly = ee.ImageCollection.fromImages(years.map(make_monthly).flatten())

ndvi_annual_mean     = ndvi_monthly.mean().clip(Aus)
ndvi_annual_mean_int = ndvi_annual_mean.toInt16()

ndvi_vis_params = {
    'min': 0,
    'max': 8000,
    'palette': ['#8B4513', '#FFFFE0', '#ADFF2F', '#228B22', '#006400']
}
print("✅ Step 5: NDVI ready")

ee.batch.Export.image.toDrive(
    image=ndvi_annual_mean,
    description='NDVI_Annual_Mean_Aus',
    folder=DRIVE_FOLDER,
    scale=1000,
    crs='EPSG:4326',
    fileFormat='GeoTIFF',
    maxPixels=1e10,
    region=australia_geometry
).start()


# ============================================================
# STEP 6 — FOREST LOSS (Hansen 2020–2022)
# ============================================================
loss_year = hansen.select(['lossyear'])
forest_loss_period = loss_year.gte(2020 - 2000).And(loss_year.lte(2022 - 2000))
forest_loss_masked = hansen.select(['loss']).updateMask(forest_loss_period).clip(Aus)
forest_loss_vis = {'min': 0, 'max': 1, 'palette': ['red']}
print("✅ Step 6: Forest Loss ready")

ee.batch.Export.image.toDrive(
    image=forest_loss_masked,
    description='Forest_Loss_Aus_2020_2022',
    folder=DRIVE_FOLDER,
    scale=500,
    crs='EPSG:4326',
    fileFormat='GeoTIFF',
    maxPixels=1e10,
    region=australia_geometry
).start()


# ============================================================
# ROBUST PNG EXPORT WITH BLACK BACKGROUND (bbox + auto-retry)
# ============================================================
def _bbox_from_geometry(geom, pad_deg=0.5):
    """Return [minLon, minLat, maxLon, maxLat] for a geometry."""
    coords = geom.bounds().getInfo()['coordinates'][0]
    xs = [c[0] for c in coords]
    ys = [c[1] for c in coords]
    return [min(xs) - pad_deg, min(ys) - pad_deg,
            max(xs) + pad_deg, max(ys) + pad_deg]


def export_png(image, vis, filename, region=australia_geometry,
               scale=2000, dimensions=None, max_retries=3,
               black_bg=True):
    """
    Robust PNG export using getThumbURL with optional black background.

    - Converts region → bbox (avoids thumbnail-service 400s).
    - Blends a solid black RGB image underneath the visualized data
      so transparent/masked pixels render as black.
    - Retries with progressively smaller payloads on failure.
    """
    vis_img = image.visualize(**vis)

    if black_bg:
        # Solid black image clipped to the same region, blended under data
        black = ee.Image.rgb(0, 0, 0).clip(region)
        vis_img = black.blend(vis_img).byte()

    bbox = _bbox_from_geometry(region)

    attempts = []
    if dimensions:
        attempts.append({'region': bbox, 'format': 'png', 'dimensions': dimensions})
    else:
        attempts.append({'region': bbox, 'format': 'png', 'scale': scale})
        attempts.append({'region': bbox, 'format': 'png', 'scale': scale * 2})
        attempts.append({'region': bbox, 'format': 'png', 'dimensions': 1024})

    last_err = None
    for i, params in enumerate(attempts[:max_retries], 1):
        try:
            url = vis_img.getThumbURL(params)
            r = requests.get(url, stream=True, timeout=300)
            r.raise_for_status()
            with open(filename, 'wb') as f:
                for chunk in r.iter_content(8192):
                    f.write(chunk)
            print(f"✅ PNG: {filename}  (attempt {i}, params={params})")
            return True
        except HTTPError as e:
            last_err = e
            print(f"⚠️ Attempt {i} failed for {filename}: {e}")
        except Exception as e:
            last_err = e
            print(f"⚠️ Attempt {i} failed for {filename}: {e}")

    print(f"❌ Could not export {filename}: {last_err}")
    return False


print("\n🎨 Exporting PNG maps (black background)...")
export_png(ba,                    burn_vis_params, f'{PNG_DIR}/Burned_Area_2021.png',      scale=2000)
export_png(merged_fire,           fire_vis_params, f'{PNG_DIR}/Fire_Categories_2023.png',  scale=2000)
export_png(fire_count,            fire_count_vis,  f'{PNG_DIR}/Fire_Count_2023.png',       scale=2000)
export_png(mean_lst,              lst_vis_params,  f'{PNG_DIR}/LST_Australia_2019.png',    scale=2000)
export_png(classified_land_cover, lulc_vis_params, f'{PNG_DIR}/LULC_Australia_2022.png',   scale=2000)
export_png(ndvi_annual_mean_int,  ndvi_vis_params, f'{PNG_DIR}/NDVI_Australia_Mean.png',   scale=2000)
export_png(forest_loss_masked,    forest_loss_vis, f'{PNG_DIR}/Forest_Loss_2020_2022.png', scale=2000)


# ============================================================
# PILLOW POST-PROCESSING — flatten + stylize with title/border
# ============================================================
def stylize_png(src_path, dst_path, title=None, bg=(0, 0, 0),
                border_px=6, border_color=(255, 255, 255),
                padding_px=20, title_h=56):
    """
    Flatten transparent PNG over a solid bg, add padding, border, and title.
    Guarantees fully opaque RGB output.
    """
    img = Image.open(src_path).convert("RGBA")

    # Flatten alpha over solid bg
    flat = Image.alpha_composite(Image.new("RGBA", img.size, bg + (255,)), img)

    # Padding
    W, H = flat.size
    canvas = Image.new("RGBA",
                       (W + 2 * padding_px, H + 2 * padding_px),
                       bg + (255,))
    canvas.paste(flat, (padding_px, padding_px), flat)

    # Border
    draw = ImageDraw.Draw(canvas)
    for i in range(border_px):
        draw.rectangle(
            [i, i, canvas.size[0] - 1 - i, canvas.size[1] - 1 - i],
            outline=border_color
        )

    # Title bar
    if title:
        try:
            font = ImageFont.truetype("DejaVuSans-Bold.ttf", 26)
        except Exception:
            font = ImageFont.load_default()

        final = Image.new("RGBA",
                          (canvas.size[0], canvas.size[1] + title_h),
                          bg + (255,))
        final.paste(canvas, (0, title_h))
        draw = ImageDraw.Draw(final)

        bbox = draw.textbbox((0, 0), title, font=font)
        tw = bbox[2] - bbox[0]
        draw.text(((final.size[0] - tw) / 2, 14),
                  title, fill=(255, 255, 255), font=font)
        canvas = final

    canvas.convert("RGB").save(dst_path, "PNG", optimize=True)
    print(f"✨ Stylized: {dst_path}")


# Titles for the 7 layers
titles = {
    "Burned_Area_2021.png":      "Burned Area — Australia 2021 (MODIS MCD64A1)",
    "Fire_Categories_2023.png":  "VIIRS Fire Temperature Classes — Oct 2023",
    "Fire_Count_2023.png":       "VIIRS Fire Detection Density — Oct 2023",
    "LST_Australia_2019.png":    "Mean Daytime Land Surface Temperature — 2019",
    "LULC_Australia_2022.png":   "Land Cover — Australia 2022 (MODIS MCD12C1)",
    "NDVI_Australia_Mean.png":   "Mean NDVI — Australia 2010–2021 (MODIS MOD13A2)",
    "Forest_Loss_2020_2022.png": "Forest Loss — Australia 2020–2022 (Hansen GFC)",
}

print("\n🖌️ Stylizing PNGs (title + border)...")
for src in sorted(glob.glob(f"{PNG_DIR}/*.png")):
    name = os.path.basename(src)
    stylize_png(
        src,
        os.path.join(PNG_FINAL_DIR, name),
        title=titles.get(name, name.replace("_", " ").replace(".png", "")),
        bg=(0, 0, 0),
        border_color=(255, 255, 255)
    )

print(f"\n🎉 Report-ready figures in ./{PNG_FINAL_DIR}/")


# ============================================================
# WEBGIS TILE URLs
# ============================================================
def get_tile_url(image, vis, name):
    map_id = image.visualize(**vis).getMapId()
    url = map_id['tile_fetcher'].url_format
    print(f"\n🗺️ {name}:\n   {url}")
    return url

print("\n" + "=" * 60)
print("WEBGIS TILE URLs (paste into Leaflet / OpenLayers)")
print("=" * 60)

tile_urls = {
    'burned':      get_tile_url(ba,                    burn_vis_params,   'Burned Area 2021'),
    'fire':        get_tile_url(merged_fire,           fire_vis_params,   'Fire Categories 2023'),
    'fire_count':  get_tile_url(fire_count,            fire_count_vis,    'Fire Detection Count 2023'),
    'lst':         get_tile_url(mean_lst,              lst_vis_params,    'LST 2019'),
    'lulc':        get_tile_url(classified_land_cover, lulc_vis_params,   'Land Cover 2022'),
    'ndvi':        get_tile_url(ndvi_annual_mean_int,  ndvi_vis_params,   'NDVI Mean'),
    'forest_loss': get_tile_url(forest_loss_masked,    forest_loss_vis,   'Forest Loss 2020–2022'),
}

with open('tile_urls.json', 'w') as f:
    json.dump(tile_urls, f, indent=2)
print("\n💾 Saved tile_urls.json")


# ============================================================
# INTERACTIVE MAP (Jupyter display)
# ============================================================
m = geemap.Map(center=[-25, 134], zoom=4)
m.add_basemap('HYBRID')

m.addLayer(ba,                    burn_vis_params,   '🔥 Burned Area 2021')
m.addLayer(mean_lst,              lst_vis_params,    '🌡️ LST 2019')
m.addLayer(classified_land_cover, lulc_vis_params,   '🌍 Land Cover 2022')
m.addLayer(ndvi_annual_mean_int,  ndvi_vis_params,   '🌿 NDVI Mean')
m.addLayer(forest_loss_masked,    forest_loss_vis,   '🌲 Forest Loss 2020–2022')
m.addLayer(merged_fire,           fire_vis_params,   '🔥 Fire Categories 2023')
m.addLayer(fire_count,            fire_count_vis,    '🔥 Fire Detection Count 2023')
m.addLayer(Aus,                   {'color': 'black'}, 'Australia Boundary')

m.addLayerControl()

legend_dict = {
    'Cool (300–320K)':     'ffff00',
    'Moderate (320–340K)': 'ffa500',
    'Hot (340–360K)':      'ff0000',
    'Very Hot (360–380K)': 'ffffff',
    'Extreme (>380K)':     '8b0000',
}
m.add_legend(title='Fire Temperature Class', legend_dict=legend_dict)

print("\n✅ ALL STEPS COMPLETE")
print(f"   📁 Raw PNGs (black bg):        ./{PNG_DIR}/")
print(f"   📁 Stylized PNGs (title+border): ./{PNG_FINAL_DIR}/")
print(f"   📁 Drive exports:              {DRIVE_FOLDER}/")
print(f"   📁 WebGIS tile URLs:           tile_urls.json")
m




✅ EE initialized
✅ Datasets loaded
✅ Step 1: Burned Area ready
✅ Step 2: VIIRS scenes in AOI/period: 22
✅ Step 2: Fire points ready (~38572 pts)
📤 Fire CSVs + GeoTIFF queued
✅ Step 3: LST ready
✅ Step 4: Land Cover ready
✅ Step 5: NDVI ready
✅ Step 6: Forest Loss ready

🎨 Exporting PNG maps (black background)...
✅ PNG: exports_png/Burned_Area_2021.png  (attempt 1, params={'region': [112.42108988316402, -44.24058444952691, 159.60954806417263, -8.642471596906729], 'format': 'png', 'scale': 2000})
✅ PNG: exports_png/Fire_Categories_2023.png  (attempt 1, params={'region': [112.42108988316402, -44.24058444952691, 159.60954806417263, -8.642471596906729], 'format': 'png', 'scale': 2000})
✅ PNG: exports_png/Fire_Count_2023.png  (attempt 1, params={'region': [112.42108988316402, -44.240584449526914, 159.60954806417263, -8.642471596906729], 'format': 'png', 'scale': 2000})
⚠️ Attempt 1 failed for exports_png/LST_Australia_2019.png: HTTPSConnectionPool(host='earthengine.googleapis.com', port=443)

Map(center=[-25, 134], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(ch…